# T3 — replica conversazionale in napoletano

Riscrittura completa. Del notebook precedente resta solo la configurazione Kaggle
(GPU T4 singola, QLoRA 4-bit, Secrets, resume via dataset montato).

## Cosa cambia, e perché

Il modello risponde in napoletano ma ignora il contesto. Non è un difetto di
prompting: è il risultato prevedibile di come sono fatti i dati e la loss.

| causa | intervento |
|---|---|
| il contesto è una finestra fissa di 3 turni ≈ 16 parole, dominata da riscontri (`mh`, `sì`) | **ricostruzione della trascrizione** dai file che hai già: 2.285 turni contro 1.127, finestra portata a 6 |
| l'istruzione è la stessa stringa in tutti gli esempi → informazione mutua ~0 col target → il modello impara a saltarla | **istruzione variabile** (8 parafrasi) + segnali derivati dal contesto (domanda / riscontro) |
| la cross-entropy su 750 target dice *«produci questa stringa»*, non *«produci una replica che ci sta»* | **loss contrastiva sul contesto**: ogni esempio è visto col contesto vero e con uno falso, e si penalizza il modello che non li distingue |
| la selezione del checkpoint su chrF premia chi indovina le parole del turno realmente pronunciato | **selezione su `ctx_delta`** = logP(target\|ctx vero) − logP(target\|ctx falso) |
| il fine turno del chat template non entra in `generation_config` → uscite 2-3× più lunghe dell'umano | **verificato prima** di addestrare, non dopo |
| nessuna misura diretta di «usa il contesto» | **ablazione** come metrica primaria: stesso item con contesto vero e falso, chrF *fra le due uscite* |

Niente stadi A / A2: la pertinenza sta nel modello di base ed è quella che il
continued pretraining erode. Se poi serve più dialettalità si aggiunge con
`--init-adapter`, come arm dichiarato, non come default.

## Ordine delle celle

1–4 ambiente e dati · 5–7 audit e costruzione dello split · 8–10 training
11–13 valutazione e decodifica contestuale · 14 salvataggio

## 1 — Ambiente

T4 = compute capability 7.5: **niente bf16 nativo**, si usa fp16. `torch.cuda.is_bf16_supported()` direbbe `True` includendo l'emulazione, per questo non lo si usa come test.

In [1]:
# --- 1. Ambiente -------------------------------------------------------------
!pip install -q peft bitsandbytes sacrebleu

import torch, importlib
for m in ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "sacrebleu"):
    try:
        print(f"{m:14s}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:14s} NON DISPONIBILE ({e})")

if not torch.cuda.is_available():
    raise SystemExit("Nessuna GPU. Settings > Accelerator > GPU T4 x2.")
cc = torch.cuda.get_device_capability(0)
print("\nGPU:", torch.cuda.get_device_name(0), f"(cc {cc[0]}.{cc[1]})",
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("bf16 nativo (serve cc >= 8):", cc[0] >= 8,
      "-> precisione:", "bf16" if cc[0] >= 8 else "fp16")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 14.3 MB/s eta 0:00:00
torch          2.11.0+cu128
transformers   5.15.0
peft           0.20.0
bitsandbytes   0.50.1
accelerate     1.14.0
sacrebleu      2.6.0

GPU: Tesla T4 (cc 7.5) | 15.6 GB
bf16 nativo (serve cc >= 8): False -> precisione: fp16


## 2 — Ambiente di esecuzione

Il notebook gira su **Kaggle** e su **Colab**: la cella qui sotto rileva dove si
trova e fissa i percorsi di conseguenza. Da qui in poi nessuna cella contiene
percorsi assoluti scritti a mano.

| | Kaggle | Colab |
|---|---|---|
| base | `/kaggle/working` | `/content` |
| dati in ingresso | `/kaggle/input` (Add Data) | upload, o Drive montato |
| persistenza | *Save Version > Save & Run All* | Drive, o download manuale |

La GPU è la stessa T4 (cc 7.5, fp16) in entrambi i casi, quindi il training non
cambia: cambia solo l'I/O.

In [2]:
# --- 2. Ambiente e percorsi --------------------------------------------------
import os, sys, pathlib

SU_KAGGLE = os.path.isdir("/kaggle/working")
SU_COLAB  = ("google.colab" in sys.modules) or os.path.isdir("/content")
BASE   = "/kaggle/working" if SU_KAGGLE else ("/content" if SU_COLAB else os.getcwd())
AMBIENTE = "Kaggle" if SU_KAGGLE else ("Colab" if SU_COLAB else "locale")

LAVORO = f"{BASE}/training"     # gli script .py
SPLIT  = f"{BASE}/split_t3"     # lo split arricchito
RUNS   = f"{BASE}/runs"         # gli adapter
EVAL   = f"{BASE}/eval"         # le generazioni e le metriche
for d in (LAVORO, RUNS, EVAL):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

os.chdir(LAVORO)
sys.path.insert(0, LAVORO)                  # perche' `from t3_train import ...` funzioni
os.environ["CUDA_VISIBLE_DEVICES"] = "0"    # una GPU sola anche su T4 x2
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"ambiente: {AMBIENTE} | base: {BASE}")
print("cwd:", os.getcwd())

ambiente: Colab | base: /content
cwd: /content/training


In [3]:
# --- 3. Token HuggingFace: FACOLTATIVO --------------------------------------
# Serve solo per i modelli gated (Llama-2, Gemma). Minerva e' pubblica: se usi
# solo quella, questa cella puo' non trovare niente e il notebook gira lo stesso.
import os

def trova_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "variabile d'ambiente"
    try:                                     # Colab: chiave a sinistra > Secrets
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t, "Colab userdata"
    except Exception:
        pass
    try:                                     # Kaggle: Add-ons > Secrets
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t:
            return t, "Kaggle Secrets"
    except Exception:
        pass
    return None, None

TOKEN, DA_DOVE = trova_token()
if TOKEN:
    os.environ["HF_TOKEN"] = TOKEN           # lo leggono anche i sottoprocessi !python
    print(f"Token trovato ({DA_DOVE}): {TOKEN[:6]}...")
else:
    print("Nessun token. Va bene per Minerva (pubblica).")
    print("Per Llama-2 o Gemma servirebbe: Colab -> chiave a sinistra > nuovo segreto")
    print("HF_TOKEN con accesso al notebook; Kaggle -> Add-ons > Secrets.")

Token trovato (Colab userdata): hf_nkE...


## 3 — Trova i file dello split

Cerca una cartella che contenga `train.json`, `dev.json` e `test.json` nei posti
plausibili dei due ambienti. Se ne trova più d'una e sceglie quella sbagliata,
imposta `IN_DIR` a mano nella cella successiva.

Su Colab i file non ci sono finché non li carichi: o li trascini nel pannello a
sinistra, o monti Drive con `from google.colab import drive; drive.mount('/content/drive')`.

In [4]:
# --- 4. Individua la cartella con train/dev/test.json di T3 ------------------
import glob, os, json

RADICI = ["/kaggle/input", "/content", "/content/drive/MyDrive", BASE, os.getcwd()]
trovati = set()
for radice in RADICI:
    if os.path.isdir(radice):
        trovati |= set(glob.glob(os.path.join(radice, "**", "train.json"), recursive=True))
candidati = sorted({os.path.dirname(p) for p in trovati
                    if all(os.path.exists(os.path.join(os.path.dirname(p), f"{s}.json"))
                           for s in ("train", "dev", "test"))})
candidati = [c for c in candidati if os.path.abspath(c) != os.path.abspath(SPLIT)]

if not candidati:
    print("Cercato in:", [r for r in RADICI if os.path.isdir(r)])
    print()
    if SU_COLAB:
        print("Carica i tre file: pannello a sinistra > icona cartella > upload,")
        print("oppure esegui in una cella:")
        print("    from google.colab import files; files.upload()")
        print("    from google.colab import drive; drive.mount('/content/drive')")
    else:
        print("Allega il dataset con Add Data.")
    raise SystemExit("Split non trovato: servono train.json, dev.json, test.json.")

IN_DIR = next((c for c in candidati if "layout3" in c or "replica" in c), candidati[-1])
print("IN_DIR =", IN_DIR)
if len(candidati) > 1:
    print("! piu' candidati:", candidati, "\n  se non e' quello giusto imposta IN_DIR a mano")

for s in ("train", "dev", "test"):
    d = json.load(open(os.path.join(IN_DIR, f"{s}.json"), encoding="utf-8"))
    print(f"  {s:5s} {len(d):5d} istanze | chiavi: {sorted(d[0])}")
    assert {"prompt", "target", "conversazione", "turn_index", "speaker"} <= set(d[0]), \
        "formato inatteso: servono prompt, target, conversazione, turn_index, speaker"

IN_DIR = /content/split/layout3_replica_conversazionale
  train   748 istanze | chiavi: ['conversazione', 'fonte', 'id', 'layout', 'prompt', 'similarita', 'speaker', 'speaker_id', 'split', 'target', 'turn_index']
  dev     187 istanze | chiavi: ['conversazione', 'fonte', 'id', 'layout', 'prompt', 'similarita', 'speaker', 'speaker_id', 'split', 'target', 'turn_index']
  test    192 istanze | chiavi: ['conversazione', 'fonte', 'id', 'layout', 'prompt', 'similarita', 'speaker', 'speaker_id', 'split', 'target', 'turn_index']


## 4 — `t3_dati.py`

Scrive lo script. Fa due cose, entrambe senza GPU: l'**audit** del corpus e la
**ricostruzione** della trascrizione.

La ricostruzione è il punto: nei file originali il campo `prompt` contiene gli
ultimi 3 turni della *trascrizione*, non gli ultimi 3 turni presenti nello split.
Le finestre di item consecutivi si sovrappongono, quindi la loro unione ricostruisce
la conversazione. Verificato sul corpus: per ogni coppia di item a distanza ≤ 3 il
target del precedente ricompare nella finestra del successivo, alla posizione attesa.

In [5]:
%%writefile t3_dati.py
#!/usr/bin/env python3
"""
t3_dati.py — audit dello split T3 e costruzione dello split arricchito.

Due funzioni, entrambe senza GPU:

  --audit         stampa le patologie del corpus con i numeri, e si ferma.
  (default)       ricostruisce la trascrizione e riscrive train/dev/test con
                  contesto piu' lungo, istruzione variabile e ancoraggio esplicito
                  al turno da riprendere.

Perche' la ricostruzione e' possibile
-------------------------------------
Nei file originali il campo `prompt` contiene gli ULTIMI 3 turni della
trascrizione, non gli ultimi 3 turni *presenti nello split*. Le finestre di item
consecutivi si sovrappongono, quindi l'unione delle finestre ricostruisce la
trascrizione: 2.285 turni contro i 1.127 usati come target. Meta' del contesto
disponibile e' gia' nei file e viene buttata via dalla finestra fissa a 3.

Uso:
    python t3_dati.py --in-dir  /kaggle/input/.../layout3_replica_conversazionale --audit
    python t3_dati.py --in-dir  /kaggle/input/.../layout3_replica_conversazionale \
                      --out-dir /kaggle/working/split_t3 --finestra 6
"""
from __future__ import annotations

import argparse
import json
import os
import random
import re
import statistics
from collections import Counter, defaultdict

SPLITS = ("train", "dev", "test")

# Riscontri / segnali di ascolto: turni che non portano contenuto proposizionale.
# Servono COME CONTESTO (dicono che l'altro sta ancora parlando) ma come TARGET
# insegnano che una replica vuota e' una replica corretta.
RISCONTRI = {
    "mh", "mhmh", "mhm", "hm", "eh", "ehm", "emh", "ah", "oh", "eeh", "uh",
    "si", "sì", "no", "boh", "okay", "ok", "esatto", "certo", "vabbuò", "vabbè",
    "cioè", "overamente", "oddio", "embè",
}


def is_riscontro(testo: str) -> bool:
    """Vero se il turno e' fatto solo di riscontri, eventualmente ripetuti."""
    parole = re.findall(r"[\w'àèéìòùâêîôû]+", testo.lower())
    if not parole:
        return True
    return all(p in RISCONTRI for p in parole)


# --------------------------------------------------------------------------- #
# I/O
# --------------------------------------------------------------------------- #

def carica(in_dir: str) -> dict[str, list[dict]]:
    dati = {}
    for s in SPLITS:
        p = os.path.join(in_dir, f"{s}.json")
        if not os.path.exists(p):
            raise SystemExit(f"manca {p}")
        dati[s] = json.load(open(p, encoding="utf-8"))
    return dati


def righe_contesto(prompt: str) -> list[tuple[str, str]]:
    """Estrae [(speaker, testo)] dal blocco di contesto del prompt originale."""
    blocco = prompt.split("---")[0]
    out = []
    for riga in blocco.strip().split("\n")[1:]:      # salta 'Conversazione finora:'
        if ":" in riga:
            sp, txt = riga.split(":", 1)
            sp = sp.strip()
            if len(sp) == 1 and sp.isalpha():
                out.append((sp.upper(), txt.strip()))
    return out


# --------------------------------------------------------------------------- #
# Ricostruzione della trascrizione
# --------------------------------------------------------------------------- #

def ricostruisci(dati: dict[str, list[dict]]):
    """conversazione -> {turn_index: (speaker, testo)}.

    L'ultima riga di contesto dell'item con indice k e' il turno k-1, la
    penultima k-2, e cosi' via. Verificato sul corpus: per ogni coppia di item
    consecutivi con distanza <= 3 il target del precedente ricompare nella
    finestra del successivo, alla posizione attesa.
    """
    turni: dict[str, dict[int, tuple[str, str]]] = defaultdict(dict)
    conflitti = 0
    for s in SPLITS:
        for x in dati[s]:
            c, k = x["conversazione"], int(x["turn_index"])
            ctx = righe_contesto(x["prompt"])
            for j, (sp, txt) in enumerate(reversed(ctx)):
                idx = k - 1 - j
                nuovo = (sp, txt)
                if idx in turni[c] and turni[c][idx] != nuovo:
                    conflitti += 1
                turni[c][idx] = nuovo
            nuovo = (x["speaker"], x["target"])
            if k in turni[c] and turni[c][k] != nuovo:
                conflitti += 1
            turni[c][k] = nuovo
    return turni, conflitti


# --------------------------------------------------------------------------- #
# Audit
# --------------------------------------------------------------------------- #

def audit(dati, turni, conflitti):
    print("=" * 74)
    print("AUDIT DELLO SPLIT T3")
    print("=" * 74)

    tot = sum(len(dati[s]) for s in SPLITS)
    print(f"\nistanze: " + " | ".join(f"{s} {len(dati[s])}" for s in SPLITS) + f"  (tot {tot})")

    # --- 1. copertura e disegno dello split --------------------------------
    print("\n[1] DISEGNO DELLO SPLIT")
    for s in SPLITS:
        per = defaultdict(list)
        for x in dati[s]:
            per[x["conversazione"]].append(x["turn_index"])
        desc = ", ".join(f"{c}: turni {min(v)}-{max(v)} ({len(v)})" for c, v in sorted(per.items()))
        print(f"  {s:5s} {desc}")
    print("  -> split CRONOLOGICO: nessuna sovrapposizione di indici fra split.")
    print("     Niente leakage di adiacenza, ma dev/test sono la CODA delle stesse")
    print("     due conversazioni: non misurano generalizzazione a parlanti o temi nuovi.")

    convs = {x["conversazione"] for s in SPLITS for x in dati[s]}
    print(f"  conversazioni totali: {len(convs)} ({sorted(convs)}) -> n=2 per qualsiasi")
    print("     intervallo di confidenza sul 'parlare in napoletano di un dato gruppo'.")

    # --- 2. contesto --------------------------------------------------------
    print("\n[2] INFORMAZIONE NEL CONTESTO  <- la criticita' principale per questo task")
    for s in SPLITS:
        n_turni, parole, tutti_riscontri = Counter(), [], 0
        for x in dati[s]:
            ctx = righe_contesto(x["prompt"])
            n_turni[len(ctx)] += 1
            parole.append(sum(len(t.split()) for _, t in ctx))
            if ctx and all(is_riscontro(t) for _, t in ctx):
                tutti_riscontri += 1
        print(f"  {s:5s} turni/contesto {dict(sorted(n_turni.items()))} | "
              f"parole medie {statistics.mean(parole):.1f} (mediana {statistics.median(parole):.0f}) | "
              f"contesti di soli riscontri {100*tutti_riscontri/len(dati[s]):.1f}%")
    freq = Counter(t for s in SPLITS for x in dati[s] for _, t in righe_contesto(x["prompt"]))
    print("  turni di contesto piu' frequenti:",
          ", ".join(f"{t!r}x{n}" for t, n in freq.most_common(8)))
    print("  -> la finestra e' fissa a 3 turni e i turni sono frammenti di parlato:")
    print("     ~16 parole di contesto in tutto, dominate da riscontri. Un modello che")
    print("     ignora un contesto quasi vuoto non sta sbagliando: sta stimando bene.")

    # --- 3. trascrizione ricostruibile -------------------------------------
    print("\n[3] CONTESTO RECUPERABILE (gratis, dai file che hai gia')")
    print(f"  conflitti nella ricostruzione: {conflitti} (0-1 atteso)")
    ric = sum(len(m) for m in turni.values())
    print(f"  turni ricostruiti: {ric} contro {tot} usati come target "
          f"({ric/tot:.1f}x materiale di contesto)")
    for c, m in sorted(turni.items()):
        ks = sorted(m)
        span = ks[-1] - ks[0] + 1
        print(f"    {c}: {len(m)} turni su uno span di {span} -> copertura {100*len(m)/span:.1f}%")
    for W in (3, 6, 8, 10):
        ok = sum(1 for s in SPLITS for x in dati[s]
                 if all((x["turn_index"] - 1 - j) in turni[x["conversazione"]] for j in range(W)))
        print(f"  item con {W:2d} turni di contesto contigui ricostruibili: "
              f"{ok}/{tot} ({100*ok/tot:.1f}%)")

    # --- 4. target ----------------------------------------------------------
    print("\n[4] TARGET")
    for s in SPLITS:
        L = [len(x["target"].split()) for x in dati[s]]
        risc = sum(1 for x in dati[s] if is_riscontro(x["target"]))
        print(f"  {s:5s} parole: media {statistics.mean(L):.1f} mediana {statistics.median(L):.0f} "
              f"max {max(L)} | <=4 parole {100*sum(1 for l in L if l<=4)/len(L):.1f}% | "
              f"target di soli riscontri {100*risc/len(dati[s]):.1f}%")
    print("  -> riferimento SINGOLO su un task open-ended: chrF/BERTScore contro l'unico")
    print("     turno realmente pronunciato premiano l'indovino, non la pertinenza.")

    # --- 5. provenienza -----------------------------------------------------
    print("\n[5] PROVENIENZA DEI RIFERIMENTI")
    for s in SPLITS:
        f = Counter(x.get("fonte", "?") for x in dati[s])
        sint = sum(v for k, v in f.items() if k != "golden")
        print(f"  {s:5s} {dict(f)}  -> {100*sint/len(dati[s]):.1f}% sintetico")
    print("  -> in TEST i riferimenti non-golden sono generati da un altro modello:")
    print("     misurare la somiglianza a un output di macchina non e' misurare la qualita'.")
    print("     Vanno esclusi dal test, o riportati come strato separato.")

    # --- 6. parlanti --------------------------------------------------------
    print("\n[6] PARLANTI")
    for s in SPLITS:
        print(f"  {s:5s} target per parlante: {dict(Counter(x['speaker'] for x in dati[s]))}")
    print("  -> KPN003 e' multi-parte (A/B/C/D). Con 3 turni di finestra l'informazione")
    print("     su CHI parla a CHI non c'e': l'indirizzamento e' irrecuperabile dal prompt.")
    print("     Il campo speaker_id esiste nei dati ma non entra mai nel prompt.")

    # --- 7. istruzione ------------------------------------------------------
    print("\n[7] ISTRUZIONE")
    istr = Counter(x["prompt"].split("---")[-1].strip() for s in SPLITS for x in dati[s])
    print(f"  varianti distinte: {len(istr)} su {tot} item -> {dict(list(istr.items())[:5])}")
    print("  -> l'istruzione varia solo per la lettera del parlante. Una feature quasi")
    print("     costante ha informazione mutua ~0 col target: il modello impara a")
    print("     saltarla e a condizionare sui soli primi token. E' il meccanismo con cui")
    print("     nasce la 'cecita' al contesto.")

    # --- 8. duplicati -------------------------------------------------------
    print("\n[8] IGIENE")
    for s in SPLITS:
        pr = [x["prompt"] for x in dati[s]]
        print(f"  {s:5s} prompt duplicati interni: {len(pr)-len(set(pr))}")
    for a in SPLITS:
        for b in SPLITS:
            if a < b:
                ov = len(set(x["prompt"] for x in dati[a]) & set(x["prompt"] for x in dati[b]))
                print(f"  prompt in comune {a}/{b}: {ov}")
    print("  -> nessun duplicato e nessuna sovrapposizione: questa parte e' pulita.")
    print("=" * 74)


# --------------------------------------------------------------------------- #
# Costruzione dello split arricchito
# --------------------------------------------------------------------------- #

ISTRUZIONI = [
    "Tocca a {sp}. Rispondi in napoletano, restando sul tema.",
    "Continua la conversazione: scrivi il turno di {sp} in napoletano.",
    "Adesso parla {sp}. Che dice, in napoletano?",
    "Sei {sp}. Rispondi in napoletano a quello che ha appena detto {prec}.",
    "Scrivi la replica di {sp} in napoletano: dev'essere una risposta a {prec}.",
    "Turno di {sp}. Rispondi in napoletano, breve e pertinente.",
    "Come risponderebbe {sp}, in napoletano?",
    "Riprendi il discorso di {prec}: parla {sp}, in napoletano.",
]


def costruisci_prompt(ctx, speaker, finestra, rng, varia=True):
    """Prompt arricchito. Tutte le feature derivano dal CONTESTO, mai dal target."""
    ctx = ctx[-finestra:]
    parlanti = sorted({sp for sp, _ in ctx} | {speaker})
    prec = ctx[-1][0] if ctx else "l'altro"

    testa = (f"Conversazione in napoletano fra {len(parlanti)} persone "
             f"({', '.join(parlanti)}).")
    corpo = "\n".join(f"{sp}: {txt}" for sp, txt in ctx) if ctx else "(inizio della conversazione)"

    # segnali calcolati dal contesto: sono disponibili anche in inferenza.
    ultimo = ctx[-1][1] if ctx else ""
    segnali = []
    if ultimo.strip().endswith("?") or re.match(
            r"^\s*(che|chi|comme|quanno|addo|pecch|quant|ma )", ultimo.lower()):
        segnali.append("l'ultimo turno e' una domanda: rispondi alla domanda")
    if ctx and is_riscontro(ultimo):
        prec_cont = next((t for _, t in reversed(ctx[:-1]) if not is_riscontro(t)), "")
        if prec_cont:
            segnali.append("l'ultimo turno e' solo un cenno di ascolto: "
                           "il tema vero e' nel turno precedente")

    istr = rng.choice(ISTRUZIONI) if varia else ISTRUZIONI[0]
    istr = istr.format(sp=speaker, prec=prec)
    if segnali:
        istr += " (" + "; ".join(segnali) + ")"

    return f"{testa}\n{corpo}\n---\n{istr}"


def costruisci(dati, turni, finestra, varia, togli_riscontri, togli_sintetici, seed):
    rng = random.Random(seed)
    out, report = {}, {}
    for s in SPLITS:
        righe, scartati_r, scartati_s, ctx_corti = [], 0, 0, 0
        for x in dati[s]:
            c, k = x["conversazione"], int(x["turn_index"])
            if togli_riscontri and s == "train" and is_riscontro(x["target"]):
                scartati_r += 1
                continue
            if togli_sintetici and s in ("dev", "test") and x.get("fonte") != "golden":
                scartati_s += 1
                continue
            ctx = []
            for j in range(finestra, 0, -1):
                t = turni[c].get(k - j)
                if t is None:
                    ctx = []          # buco: riparti, il contesto dev'essere contiguo
                    continue
                ctx.append(t)
            if len(ctx) < min(3, finestra):
                ctx_corti += 1
            y = dict(x)
            y["prompt"] = costruisci_prompt(ctx, x["speaker"], finestra, rng, varia)
            y["n_turni_contesto"] = len(ctx)
            righe.append(y)
        out[s] = righe
        report[s] = dict(n=len(righe), scartati_riscontri=scartati_r,
                         scartati_sintetici=scartati_s, contesti_corti=ctx_corti,
                         turni_contesto_medi=round(
                             statistics.mean([r["n_turni_contesto"] for r in righe]), 2))
    return out, report


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in-dir", required=True)
    ap.add_argument("--out-dir", default="")
    ap.add_argument("--finestra", type=int, default=6)
    ap.add_argument("--audit", action="store_true")
    ap.add_argument("--no-variazioni", action="store_true")
    ap.add_argument("--tieni-riscontri", action="store_true")
    ap.add_argument("--tieni-sintetici", action="store_true")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    dati = carica(a.in_dir)
    turni, conflitti = ricostruisci(dati)

    if a.audit or not a.out_dir:
        audit(dati, turni, conflitti)
        return

    out, report = costruisci(dati, turni, a.finestra, not a.no_variazioni,
                             not a.tieni_riscontri, not a.tieni_sintetici, a.seed)
    os.makedirs(a.out_dir, exist_ok=True)
    for s in SPLITS:
        with open(os.path.join(a.out_dir, f"{s}.json"), "w", encoding="utf-8") as f:
            json.dump(out[s], f, ensure_ascii=False, indent=1)
    json.dump(report, open(os.path.join(a.out_dir, "report.json"), "w"), indent=2)

    print(f"scritto in {a.out_dir}")
    for s in SPLITS:
        print(f"  {s:5s} {report[s]}")
    print("\nesempio di prompt (train):\n" + "-" * 66)
    print(out["train"][5]["prompt"])
    print("--- TARGET:", out["train"][5]["target"])


if __name__ == "__main__":
    main()

Writing t3_dati.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5 — Audit del corpus

**Cosa fare:** esegui e leggi. Non addestra niente e non scrive niente: sono i
numeri su cui poggia tutto il resto del notebook.

In [6]:
# --- 5. Audit (nessuna GPU) --------------------------------------------------
!python t3_dati.py --in-dir {IN_DIR} --audit

AUDIT DELLO SPLIT T3

istanze: train 748 | dev 187 | test 192  (tot 1127)

[1] DISEGNO DELLO SPLIT
  train KPN001: turni 2-764 (294), KPN003: turni 2-1022 (454)
  dev   KPN001: turni 770-932 (83), KPN003: turni 1030-1242 (104)
  test  KPN001: turni 939-1102 (80), KPN003: turni 1248-1465 (112)
  -> split CRONOLOGICO: nessuna sovrapposizione di indici fra split.
     Niente leakage di adiacenza, ma dev/test sono la CODA delle stesse
     due conversazioni: non misurano generalizzazione a parlanti o temi nuovi.
  conversazioni totali: 2 (['KPN001', 'KPN003']) -> n=2 per qualsiasi
     intervallo di confidenza sul 'parlare in napoletano di un dato gruppo'.

[2] INFORMAZIONE NEL CONTESTO  <- la criticita' principale per questo task
  train turni/contesto {1: 2, 2: 2, 3: 744} | parole medie 17.7 (mediana 15) | contesti di soli riscontri 0.4%
  dev   turni/contesto {1: 1, 3: 186} | parole medie 15.7 (mediana 15) | contesti di soli riscontri 0.0%
  test  turni/contesto {1: 2, 2: 2, 3: 188} | p

## 6 — Costruisci lo split arricchito

Quattro modifiche, tutte senza dati nuovi:

- `--finestra 6` — contesto contiguo ricostruito, media ~5,5 turni contro 3
- istruzione campionata fra 8 parafrasi, più segnali **derivati dal contesto**
  (l'ultimo turno è una domanda / è un riscontro). Sono funzioni del prompt, non
  del target: disponibili identiche in inferenza, quindi non sono leakage
- target di soli riscontri tolti dal **train** (restano come contesto): insegnano
  che una replica senza contenuto è corretta
- riferimenti sintetici (`fonte != golden`) tolti da **dev/test**: misurare la
  somiglianza all'output di un altro modello non misura la qualità

Con `--tieni-riscontri` / `--tieni-sintetici` / `--no-variazioni` produci gli arm
di ablazione, uno alla volta.

In [7]:
# --- 6. Split arricchito -----------------------------------------------------
!python t3_dati.py --in-dir {IN_DIR} --out-dir {SPLIT} --finestra 6 --seed 42

import json
rep = json.load(open(f"{SPLIT}/report.json"))
for s, info in rep.items():
    print(f"  {s:5s} {info}")

scritto in /content/split_t3
  train {'n': 742, 'scartati_riscontri': 6, 'scartati_sintetici': 0, 'contesti_corti': 4, 'turni_contesto_medi': 5.47}
  dev   {'n': 165, 'scartati_riscontri': 0, 'scartati_sintetici': 22, 'contesti_corti': 1, 'turni_contesto_medi': 5.75}
  test  {'n': 168, 'scartati_riscontri': 0, 'scartati_sintetici': 24, 'contesti_corti': 3, 'turni_contesto_medi': 5.54}

esempio di prompt (train):
------------------------------------------------------------------
Conversazione in napoletano fra 3 persone (A, B, C).
B: o sti ffoto nun l'avimmo maje fatte
A: tu nun 'e vuò fà
A: sempe ca nun te vuò fà taggà da nisciuna parte
B: embè, hê miso na cosa mia ll'atu juorno
A: bell'
A: eh, sempe ca
---
Adesso parla C. Che dice, in napoletano?
--- TARGET: ecco guagliò, a vvuje
  train {'n': 742, 'scartati_riscontri': 6, 'scartati_sintetici': 0, 'contesti_corti': 4, 'turni_contesto_medi': 5.47}
  dev   {'n': 165, 'scartati_riscontri': 0, 'scartati_sintetici': 22, 'contesti_corti': 1

## 7 — `t3_train.py`

QLoRA 4-bit su T4, con la **loss contrastiva sul contesto**:

$$\mathcal{L} = \mathrm{CE}(y \mid c) + \lambda \cdot \max\big(0,\; m - [\log P(y \mid c) - \log P(y \mid \tilde{c})]\big)$$

dove $\tilde c$ è il contesto di un altro punto della conversazione. È l'unico
termine che rende il contesto **causalmente rilevante** per il gradiente: senza,
il modello minimizza la loss ignorandolo. Non serve un modello di riferimento
(non è DPO) né preferenze annotate.

Costa un secondo forward per micro-batch: circa il doppio del tempo.
`--lambda-ctx 0` la spegne ed è l'arm di confronto.

In [8]:
%%writefile t3_train.py
#!/usr/bin/env python3
"""
t3_train.py — SFT di T3 (replica conversazionale in napoletano) su Kaggle T4.

Differenze rispetto alla versione precedente, tutte mirate alla cecita' al
contesto:

  1. LOSS CONTRASTIVA SUL CONTESTO (--lambda-ctx, il pezzo nuovo).
     La cross-entropy su ~750 target non puo' insegnare la pertinenza: per un
     contesto dato ci sono centinaia di repliche valide, quindi il gradiente
     dice "produci QUESTA stringa" quando la cosa da imparare e' "produci una
     replica che ci sta". Qui ogni esempio viene visto DUE volte, col contesto
     vero e con un contesto preso da un altro punto, e si aggiunge

         relu( margine - ( logP(target|ctx vero) - logP(target|ctx falso) ) )

     Non serve un modello di riferimento (non e' DPO) e non servono preferenze
     annotate: il negativo si costruisce dai dati che ci sono. E' l'unico
     termine che rende il contesto causalmente rilevante per la loss.

  2. SELEZIONE DEL CHECKPOINT SU ctx_delta, non su chrF.
     chrF contro l'unico turno realmente pronunciato premia chi indovina quelle
     parole. ctx_delta misura esattamente cio' che manca.

  3. Fine turno verificato prima di partire, non dopo.
  4. Nessuno stadio A/A2: si parte dal modello di base. La pertinenza sta nel
     modello di base ed e' quello che il continued pretraining erode; se serve
     dialettalita' in piu' si aggiunge dopo, con --init-adapter.

Uso:
    python t3_train.py --model minerva --split-dir /kaggle/working/split_t3
"""
from __future__ import annotations

import argparse
import json
import math
import os
import random
import shutil
import sys
import time

MODEL_REGISTRY = {
    "minerva":       "sapienzanlp/Minerva-7B-instruct-v1.0",
    "llama":         "meta-llama/Llama-2-7b-chat-hf",
    "gemma":         "google/gemma-3-4b-it",
    "minerva-small": "sapienzanlp/Minerva-3B-base-v1.0",
    "gemma-tiny":    "google/gemma-3-1b-it",
}
VISION_MARKERS = ("vision_tower", "vision_model", "multi_modal_projector",
                  "visual", "image_encoder", "patch_embed")


# --------------------------------------------------------------------------- #
# Utility condivise con lo script di valutazione
# --------------------------------------------------------------------------- #

def resolve_model(nome: str) -> str:
    return MODEL_REGISTRY.get(nome, nome)


def slug(repo_id: str) -> str:
    return repo_id.split("/")[-1]


def carica_token(esplicito=None):
    """Facoltativo: serve solo per i modelli gated (Llama-2, Gemma).
    Ordine: argomento -> variabile d'ambiente -> Colab userdata -> Kaggle Secrets."""
    if esplicito:
        return esplicito
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def base_ambiente():
    """/kaggle/working su Kaggle, /content su Colab, la cwd altrove."""
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working"
    if os.path.isdir("/content"):
        return "/content"
    return os.getcwd()


def carica_split(split_dir, nome, max_samples=None):
    righe = json.load(open(os.path.join(split_dir, f"{nome}.json"), encoding="utf-8"))
    return righe[:max_samples] if max_samples else righe


def blocco_contesto(prompt: str):
    """(testa, contesto, istruzione). Il formato e' quello di t3_dati.py:
    riga di testa, righe di contesto, '---', istruzione."""
    testa_e_ctx, _, istr = prompt.partition("\n---\n")
    righe = testa_e_ctx.split("\n")
    return righe[0], righe[1:], istr


def sostituisci_contesto(prompt: str, altro: str) -> str:
    """Stesso item, contesto di un altro punto della conversazione.
    Testa e istruzione restano identiche: l'unica variabile e' il contenuto."""
    testa, _, istr = blocco_contesto(prompt)
    _, ctx_altro, _ = blocco_contesto(altro)
    return "\n".join([testa] + ctx_altro) + "\n---\n" + istr


def render(tokenizer, prompt, target=None):
    """Chat template del tokenizer. Nessun ruolo system: l'istruzione e' gia'
    dentro il prompt, byte-identica fra training e inferenza."""
    if not getattr(tokenizer, "chat_template", None):
        if target is None:
            return prompt + "\n"
        return prompt + "\n" + target + (tokenizer.eos_token or "")

    def applica(u, a):
        msg = [{"role": "user", "content": u}]
        if a is None:
            return tokenizer.apply_chat_template(msg, tokenize=False,
                                                 add_generation_prompt=True)
        return tokenizer.apply_chat_template(msg + [{"role": "assistant", "content": a}],
                                             tokenize=False, add_generation_prompt=False)
    try:
        return applica(prompt, target)
    except Exception:
        wrap = lambda s: [{"type": "text", "text": s}]
        return applica(wrap(prompt), None if target is None else wrap(target))


def target_lora(model):
    """Solo il language model: su Gemma-3 il vision tower ha moduli omonimi."""
    import torch.nn as nn
    nomi, saltati = set(), 0
    for nome, mod in model.named_modules():
        if not isinstance(mod, nn.Linear) and mod.__class__.__name__ not in (
                "Linear4bit", "Linear8bitLt"):
            continue
        foglia = nome.split(".")[-1]
        if foglia in ("lm_head",):
            continue
        if any(m in nome for m in VISION_MARKERS):
            saltati += 1
            continue
        if foglia.endswith("_proj") or foglia in ("gate_proj", "up_proj", "down_proj"):
            nomi.add(foglia)
    if saltati:
        print(f"  esclusi {saltati} moduli del vision tower")
    return sorted(nomi)


def training_args_compatibili(TrainingArguments, **kw):
    """TrainingArguments cambia firma fra transformers 4.x e 5.x (`group_by_length`,
    per esempio, in 5.x non esiste piu'). Invece di inseguire i singoli argomenti,
    si tengono solo quelli che la classe installata accetta davvero e si dichiara
    a schermo cosa e' stato scartato: un argomento silenziosamente ignorato e'
    peggio di uno mancante."""
    import inspect
    ammessi = set(inspect.signature(TrainingArguments.__init__).parameters)
    try:                                     # dataclass: i campi sono la fonte vera
        import dataclasses
        ammessi |= {f.name for f in dataclasses.fields(TrainingArguments)}
    except Exception:
        pass
    tenuti = {k: v for k, v in kw.items() if k in ammessi}
    scartati = sorted(set(kw) - set(tenuti))
    if scartati:
        print(f"  TrainingArguments: argomenti non supportati da questa versione "
              f"di transformers, ignorati -> {', '.join(scartati)}")
    return TrainingArguments(**tenuti)


def id_fine_turno(tokenizer, esempio_prompt, esempio_target):
    """L'id che il modello deve emettere per chiudere il turno. Con i chat
    template moderni NON e' eos_token_id ma <|im_end|> / <end_of_turn>: se non
    entra in generation_config.eos_token_id, generate() non si ferma mai e le
    uscite escono 2-3 volte piu' lunghe dell'umano."""
    ids = tokenizer(render(tokenizer, esempio_prompt, esempio_target),
                    add_special_tokens=False)["input_ids"]
    return ids[-1]


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #

class DatasetT3:
    """prompt (mascherato a -100) + target. Con contrastivo=True ogni item
    porta anche la versione col contesto sbagliato."""

    def __init__(self, righe, tokenizer, max_len, contrastivo=False, seed=0):
        self.righe, self.tok, self.max_len = righe, tokenizer, max_len
        self.contrastivo = contrastivo
        self.n_troncati = self.n_prefix_mismatch = 0
        rng = random.Random(seed)
        self.neg = []
        for i, r in enumerate(righe):
            if contrastivo and len(righe) > 1:
                j = rng.randrange(len(righe) - 1)
                j = j + (j >= i)
                self.neg.append(sostituisci_contesto(r["prompt"], righe[j]["prompt"]))
            else:
                self.neg.append(None)
            if i < 200:
                p = tokenizer(render(tokenizer, r["prompt"]),
                              add_special_tokens=False)["input_ids"]
                f = tokenizer(render(tokenizer, r["prompt"], r["target"]),
                              add_special_tokens=False)["input_ids"]
                if len(f) > max_len:
                    self.n_troncati += 1
                if f[:len(p)] != p:
                    self.n_prefix_mismatch += 1

    def __len__(self):
        return len(self.righe)

    def _codifica(self, prompt, target):
        p = self.tok(render(self.tok, prompt), add_special_tokens=False)["input_ids"]
        f = self.tok(render(self.tok, prompt, target),
                     add_special_tokens=False)["input_ids"][:self.max_len]
        lab = list(f)
        for j in range(min(len(p), len(lab))):
            lab[j] = -100
        return f, lab

    def __getitem__(self, i):
        r = self.righe[i]
        ids, lab = self._codifica(r["prompt"], r["target"])
        out = {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": lab}
        if self.contrastivo:
            n_ids, n_lab = self._codifica(self.neg[i], r["target"])
            out |= {"neg_input_ids": n_ids, "neg_attention_mask": [1] * len(n_ids),
                    "neg_labels": n_lab}
        return out


def make_collate(pad_id, contrastivo):
    import torch

    def pad(seqs, valore):
        n = max(len(s) for s in seqs)
        return torch.tensor([s + [valore] * (n - len(s)) for s in seqs], dtype=torch.long)

    def collate(batch):
        out = {"input_ids": pad([b["input_ids"] for b in batch], pad_id),
               "attention_mask": pad([b["attention_mask"] for b in batch], 0),
               "labels": pad([b["labels"] for b in batch], -100)}
        if contrastivo and "neg_input_ids" in batch[0]:
            out |= {"neg_input_ids": pad([b["neg_input_ids"] for b in batch], pad_id),
                    "neg_attention_mask": pad([b["neg_attention_mask"] for b in batch], 0),
                    "neg_labels": pad([b["neg_labels"] for b in batch], -100)}
        return out
    return collate


# --------------------------------------------------------------------------- #
# Loss contrastiva
# --------------------------------------------------------------------------- #

def logp_per_sequenza(logits, labels):
    """(nll_somma_totale, n_token, logp_medio_per_sequenza).

    Si indicizzano PRIMA le posizioni etichettate: castare a fp32 tutti i logit
    di un batch 4x384x32000 sono ~200 MB per forward, e qui i forward sono due.
    Le posizioni che contano sono la decina di token del target.
    """
    import torch
    import torch.nn.functional as F
    lg = logits[:, :-1, :]
    lb = labels[:, 1:]
    maschera = lb != -100
    idx_b, idx_t = maschera.nonzero(as_tuple=True)
    if idx_b.numel() == 0:
        zero = logits.sum() * 0.0
        return zero, torch.tensor(1, device=logits.device), zero.expand(logits.size(0))
    nll = F.cross_entropy(lg[idx_b, idx_t].float(), lb[idx_b, idx_t], reduction="none")
    somma = torch.zeros(logits.size(0), device=logits.device, dtype=nll.dtype
                        ).index_add_(0, idx_b, nll)
    n_seq = torch.zeros_like(somma).index_add_(0, idx_b, torch.ones_like(nll)).clamp(min=1)
    return nll.sum(), maschera.sum(), -(somma / n_seq)


def costruisci_trainer_contrastivo(Trainer, lam, margine):
    class TrainerContrastivo(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False,
                         num_items_in_batch=None, **kw):
            neg = {k[4:]: inputs.pop(k) for k in list(inputs) if k.startswith("neg_")}
            out = model(input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])
            nll_somma, n_tok, logp_pos = logp_per_sequenza(out.logits, inputs["labels"])

            # Denominatore identico per i due termini: transformers non divide
            # la loss per gli step di accumulo quando il forward accetta **kwargs
            # (un PeftModel lo fa sempre), quindi la normalizzazione va fatta qui.
            denom = num_items_in_batch if num_items_in_batch else n_tok
            loss = nll_somma / denom

            if lam > 0 and neg:
                out_n = model(input_ids=neg["input_ids"],
                              attention_mask=neg["attention_mask"])
                _, _, logp_neg = logp_per_sequenza(out_n.logits, neg["labels"])
                margine_loss = (margine - (logp_pos - logp_neg)).clamp(min=0).mean()
                loss = loss + lam * margine_loss * (n_tok / denom)

            return (loss, out) if return_outputs else loss
    return TrainerContrastivo


# --------------------------------------------------------------------------- #
# Metriche di valutazione periodica
# --------------------------------------------------------------------------- #

def callback_contesto(TrainerCallback, torch, tokenizer, righe, n, seed, max_len):
    """ctx_delta = media di logP(target|ctx vero) - logP(target|ctx falso), in
    nat per token. Sopra 0 il contesto viene usato; a 0 il modello e' cieco.
    Nessuna generazione: costa due forward su n item."""
    rng = random.Random(seed)
    campione = list(righe)
    rng.shuffle(campione)
    campione = campione[:n]
    falsi = [sostituisci_contesto(r["prompt"], campione[(i + 1) % len(campione)]["prompt"])
             for i, r in enumerate(campione)]

    class CB(TrainerCallback):
        def on_evaluate(self, args, state, control, model=None, metrics=None, **kw):
            if model is None or metrics is None:
                return
            model.eval()
            deltas, vinti = [], 0
            with torch.no_grad():
                for r, falso in zip(campione, falsi):
                    lp = []
                    for prm in (r["prompt"], falso):
                        ids = tokenizer(render(tokenizer, prm, r["target"]),
                                        add_special_tokens=False,
                                        return_tensors="pt")["input_ids"][:, :max_len]
                        n_p = len(tokenizer(render(tokenizer, prm),
                                            add_special_tokens=False)["input_ids"])
                        lab = ids.clone()
                        lab[:, :n_p] = -100
                        ids, lab = ids.to(model.device), lab.to(model.device)
                        logits = model(input_ids=ids).logits
                        lp.append(logp_per_sequenza(logits, lab)[2].item())
                    deltas.append(lp[0] - lp[1])
                    vinti += lp[0] > lp[1]
            metrics["eval_ctx_delta"] = round(sum(deltas) / len(deltas), 4)
            metrics["eval_ctx_acc"] = round(vinti / len(deltas), 4)
            print(f"    ctx_delta {metrics['eval_ctx_delta']:+.4f} nat/token | "
                  f"ctx_acc {metrics['eval_ctx_acc']:.3f} (caso: 0.500)")
            model.train()
    return CB


def callback_generazione(TrainerCallback, torch, tokenizer, righe, n, eot_ids, max_new):
    """chrF++ su generazione REALE + rapporto di lunghezza. In teacher forcing
    l'ipotesi ha per costruzione la lunghezza del riferimento, quindi il
    fallimento sull'EOS resta invisibile: qui no."""
    import sacrebleu
    campione = righe[:n]

    class CB(TrainerCallback):
        def on_evaluate(self, args, state, control, model=None, metrics=None, **kw):
            if model is None or metrics is None:
                return
            model.eval()
            model.config.use_cache = True
            ipo, rif = [], []
            with torch.no_grad():
                for r in campione:
                    ids = tokenizer(render(tokenizer, r["prompt"]),
                                    add_special_tokens=False, return_tensors="pt"
                                    ).to(model.device)
                    o = model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                                       eos_token_id=eot_ids,
                                       pad_token_id=tokenizer.pad_token_id,
                                       repetition_penalty=1.15, no_repeat_ngram_size=3)
                    ipo.append(tokenizer.decode(o[0][ids["input_ids"].shape[1]:],
                                                skip_special_tokens=True).strip())
                    rif.append(r["target"])
            model.config.use_cache = False
            metrics["eval_gen_chrf"] = round(
                sacrebleu.corpus_chrf(ipo, [rif], word_order=2).score, 3)
            lr_ = (sum(len(h.split()) for h in ipo) /
                   max(1, sum(len(t.split()) for t in rif)))
            metrics["eval_rapporto_lunghezza"] = round(lr_, 3)
            print(f"    gen_chrf {metrics['eval_gen_chrf']:.2f} | "
                  f"lunghezza generato/umano {lr_:.2f}x (obiettivo ~1.0)")
            print(f"    esempio: {ipo[0][:90]!r}\n       umano: {rif[0][:90]!r}")
            model.train()
    return CB


# --------------------------------------------------------------------------- #
# main
# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="minerva")
    ap.add_argument("--split-dir", required=True)
    ap.add_argument("--out-dir", default="")
    ap.add_argument("--tag", default="T3")
    ap.add_argument("--epochs", type=float, default=4)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--grad-accum", type=int, default=4)
    ap.add_argument("--eval-batch-size", type=int, default=4)
    ap.add_argument("--max-seq-len", type=int, default=384)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--lora-dropout", type=float, default=0.05)
    ap.add_argument("--lambda-ctx", type=float, default=0.5,
                    help="peso della loss contrastiva sul contesto (0 = spenta)")
    ap.add_argument("--margine", type=float, default=0.15,
                    help="margine in nat/token fra contesto vero e falso")
    ap.add_argument("--metric", default="ctx_delta",
                    choices=["ctx_delta", "gen_chrf", "loss"])
    ap.add_argument("--evals-per-epoch", type=int, default=2)
    ap.add_argument("--patience", type=int, default=4)
    ap.add_argument("--ctx-n", type=int, default=64)
    ap.add_argument("--gen-n", type=int, default=24)
    ap.add_argument("--gen-max-new", type=int, default=32)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--init-adapter", default="")
    ap.add_argument("--resume-dir", default="")
    ap.add_argument("--max-train-samples", type=int, default=0)
    ap.add_argument("--eval-subset", type=int, default=96)
    ap.add_argument("--no-gradient-checkpointing", action="store_true")
    ap.add_argument("--precision", default="qlora4bit", choices=["qlora4bit", "fp16"])
    a = ap.parse_args()
    a.out_dir = a.out_dir or os.path.join(base_ambiente(), "runs")

    import torch
    from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                              EarlyStoppingCallback, Trainer, TrainerCallback,
                              TrainingArguments)
    try:
        from transformers.trainer_utils import get_last_checkpoint
    except ImportError:                      # transformers 5.x
        from transformers.trainer_callback import get_last_checkpoint
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    random.seed(a.seed)
    torch.manual_seed(a.seed)

    repo = resolve_model(a.model)
    cc = torch.cuda.get_device_capability(0)
    bf16_ok = cc[0] >= 8                      # T4 = cc 7.5 -> fp16
    dtype = torch.bfloat16 if bf16_ok else torch.float16
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    import transformers
    print(f"{repo} | GPU {torch.cuda.get_device_name(0)} {vram:.1f} GB | "
          f"precisione {'bf16' if bf16_ok else 'fp16'} | "
          f"transformers {transformers.__version__}")

    token = carica_token()
    tokenizer = AutoTokenizer.from_pretrained(repo, token=token, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- dati ---------------------------------------------------------------
    train_rows = carica_split(a.split_dir, "train", a.max_train_samples or None)
    dev_rows = carica_split(a.split_dir, "dev")
    print(f"dati: train {len(train_rows)} | dev {len(dev_rows)}")

    # --- fine turno: si controlla PRIMA di addestrare ------------------------
    eot = id_fine_turno(tokenizer, train_rows[0]["prompt"], train_rows[0]["target"])
    eot_ids = sorted({eot, tokenizer.eos_token_id} - {None})
    print(f"fine turno: id={eot} repr={tokenizer.convert_ids_to_tokens([eot])[0]!r} | "
          f"eos_token_id del tokenizer={tokenizer.eos_token_id} | "
          f"generate() si fermera' su {eot_ids}")

    # --- modello ------------------------------------------------------------
    kw = dict(device_map={"": 0}, token=token, low_cpu_mem_usage=True,
              attn_implementation="eager")
    if a.precision == "qlora4bit":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    try:
        model = AutoModelForCausalLM.from_pretrained(repo, dtype=dtype, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=dtype, **kw)

    usa_ckpt = not a.no_gradient_checkpointing
    model.config.use_cache = False
    if a.precision == "qlora4bit":
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=usa_ckpt,
            gradient_checkpointing_kwargs={"use_reentrant": False} if usa_ckpt else None)
    elif usa_ckpt:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})

    if a.init_adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, a.init_adapter, is_trainable=True)
        print(f"adapter iniziale: {a.init_adapter}")
    else:
        model = get_peft_model(model, LoraConfig(
            r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=a.lora_dropout,
            target_modules=target_lora(model), bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    if usa_ckpt:
        model.enable_input_require_grads()

    # --- dataset ------------------------------------------------------------
    contrastivo = a.lambda_ctx > 0
    train_ds = DatasetT3(train_rows, tokenizer, a.max_seq_len, contrastivo, a.seed)
    dev_eval = dev_rows[:a.eval_subset] if a.eval_subset else dev_rows
    dev_ds = DatasetT3(dev_eval, tokenizer, a.max_seq_len, False, a.seed)
    if train_ds.n_prefix_mismatch:
        sys.exit(f"PREFIX MISMATCH su {train_ds.n_prefix_mismatch}/200: il prompt "
                 f"renderizzato non e' prefisso del testo completo, la maschera della "
                 f"loss e' disallineata. Non proseguire.")
    if train_ds.n_troncati:
        print(f"  ! {train_ds.n_troncati}/200 troncati a {a.max_seq_len}: alza --max-seq-len")

    # --- cadenza ------------------------------------------------------------
    eff = a.batch_size * a.grad_accum
    step_epoca = max(1, math.ceil(len(train_rows) / eff))
    max_steps = max(1, math.ceil(a.epochs * step_epoca))
    eval_steps = max(5, step_epoca // max(1, a.evals_per_epoch))
    n_eval = max_steps // eval_steps
    print(f"batch efficace {eff} | {step_epoca} step/epoca | {max_steps} totali | "
          f"eval ogni {eval_steps} -> ~{n_eval} valutazioni")
    if n_eval < a.patience + 1:
        print(f"  ! ~{n_eval} eval contro patience={a.patience}: l'early stopping "
              f"non potra' scattare.")

    run = f"{slug(repo)}__{a.tag}"
    out_dir = os.path.join(a.out_dir, run)
    os.makedirs(out_dir, exist_ok=True)
    if a.resume_dir:
        src = os.path.join(a.resume_dir, run)
        src = src if os.path.isdir(src) else a.resume_dir
        if os.path.isdir(src):
            for nome in os.listdir(src):
                if nome.startswith("checkpoint-") and not os.path.exists(
                        os.path.join(out_dir, nome)):
                    shutil.copytree(os.path.join(src, nome), os.path.join(out_dir, nome))
    riprendi = get_last_checkpoint(out_dir)
    if riprendi:
        print("riprendo da", riprendi)

    targs = training_args_compatibili(
        TrainingArguments,
        output_dir=out_dir, num_train_epochs=a.epochs,
        per_device_train_batch_size=a.batch_size,
        per_device_eval_batch_size=a.eval_batch_size,
        gradient_accumulation_steps=a.grad_accum,
        learning_rate=a.lr, lr_scheduler_type="cosine",
        warmup_steps=max(1, round(0.05 * max_steps)),
        weight_decay=0.01, max_grad_norm=0.3,
        eval_strategy="steps", eval_steps=eval_steps,
        save_strategy="steps", save_steps=eval_steps, save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=a.metric,
        greater_is_better=(a.metric != "loss"),
        logging_steps=max(1, eval_steps // 4), report_to="none",
        seed=a.seed, data_seed=a.seed,
        fp16=not bf16_ok, bf16=bf16_ok,
        optim="paged_adamw_8bit", remove_unused_columns=False,
        group_by_length=False, dataloader_num_workers=2,
        label_names=["labels"],
        # senza questo il Trainer accumula i logit di tutto il dev per
        # compute_metrics (96 x 384 x |V|): OOM garantito su T4.
        prediction_loss_only=True,
    )

    # ORDINE: EarlyStoppingCallback legge metrics[metric_for_best_model], quindi
    # deve stare DOPO i callback che scrivono ctx_delta e gen_chrf.
    cbs = []
    if a.ctx_n:
        cbs.append(callback_contesto(TrainerCallback, torch, tokenizer, dev_rows,
                                     a.ctx_n, a.seed, a.max_seq_len)())
    if a.gen_n:
        cbs.append(callback_generazione(TrainerCallback, torch, tokenizer, dev_rows,
                                        a.gen_n, eot_ids, a.gen_max_new)())
    cbs.append(EarlyStoppingCallback(early_stopping_patience=a.patience))

    Base = costruisci_trainer_contrastivo(Trainer, a.lambda_ctx, a.margine)
    trainer = Base(model=model, args=targs, train_dataset=train_ds, eval_dataset=dev_ds,
                   data_collator=make_collate(tokenizer.pad_token_id, contrastivo),
                   callbacks=cbs)

    print(f"\ncontrastiva: lambda={a.lambda_ctx} margine={a.margine} | "
          f"selezione su {a.metric}\n")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=riprendi)
    minuti = (time.time() - t0) / 60

    finale = os.path.join(out_dir, "adapter_final")
    trainer.model.save_pretrained(finale)
    tokenizer.save_pretrained(finale)

    json.dump({
        "repo_id": repo, "run": run, "minuti": round(minuti, 1),
        "split_dir": a.split_dir, "train": len(train_rows), "dev": len(dev_rows),
        "iperparametri": vars(a),
        "fine_turno": {"id": eot, "eos_usati": eot_ids},
        "best": {"step": trainer.state.best_model_checkpoint,
                 "metrica": a.metric, "valore": trainer.state.best_metric},
        "storia": [{k: v for k, v in h.items() if k.startswith("eval_") or k == "step"}
                   for h in trainer.state.log_history if "eval_loss" in h],
    }, open(os.path.join(out_dir, "summary.json"), "w"), indent=2, default=str)

    print(f"\nfatto in {minuti:.1f} min -> {finale}")
    print(f"best {a.metric} = {trainer.state.best_metric}")


if __name__ == "__main__":
    main()

Writing t3_train.py


## 8 — Smoke test

Due minuti su un modello piccolo, prima di scaricare un 7B. Tre righe da guardare:

1. `fine turno: id=... repr=...` — se il `repr` è `<|im_end|>` / `<end_of_turn>`
   e non `</s>`, il fix serviva
2. `PREFIX MISMATCH` — se compare, lo script si ferma: il masking della loss
   sarebbe disallineato
3. `ctx_delta` nella prima eval — a `0.0000` esatto il contesto non arriva al
   modello e nessun intervento a valle serve

In [9]:
# --- 8. Smoke test -----------------------------------------------------------
!python t3_train.py --model minerva-small --split-dir {SPLIT} \
    --max-train-samples 32 --eval-subset 16 --epochs 1 \
    --ctx-n 8 --gen-n 4 --evals-per-epoch 2 --tag SMOKE --out-dir {RUNS}

sapienzanlp/Minerva-3B-base-v1.0 | GPU Tesla T4 15.6 GB | precisione fp16 | transformers 5.15.0
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/sapienzanlp/Minerva-3B-base-v1.0/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

huggingface_hub.errors.GatedRepoError: 403 Client Error. (Request ID: Root=1-6a8c1cb5-3b869943667fdc472095f277;255e6c78-8cc4-47ca-b744-77706cfd8536)

Cannot access gated repo for url https://huggingface.co/sapienzanlp/Minerva-3B-base-v1.0/resolve/main/config.json.
Access to model sapienzanlp/Minerva-3B-base-v1.0 is restricted and you are not in the authorized list. Visit https://huggingface.co/sapienzanlp/Minerva-3B-base-v1.0 to ask for access.

The above exception was the direct cause of the following exception:

OSError: You are trying to access a gated repo.
Make sure to have access to it at

## 9 — Training

~1,5–2 h su T4 con `--lambda-ctx 0.5` (il doppio forward raddoppia il tempo).
Se sfori: `--batch-size 2 --grad-accum 8`, oppure `--ctx-n 32`.

Le due righe da leggere ad ogni eval:

- `ctx_delta` — deve **salire**. È la metrica di selezione del checkpoint.
- `lunghezza generato/umano` — deve stare verso **1.0**. Sopra 2 il modello non
  ha imparato a chiudere il turno e ogni altra metrica è contaminata.

In [10]:
# --- 9. Training -------------------------------------------------------------
MODEL = "minerva"
SEED  = 42

!python t3_train.py --model {MODEL} --split-dir {SPLIT} --seed {SEED} \
    --epochs 4 --lr 1e-4 --lora-r 16 \
    --lambda-ctx 0.5 --margine 0.15 \
    --metric ctx_delta --patience 4 \
    --batch-size 4 --grad-accum 4 --max-seq-len 384 \
    --ctx-n 64 --gen-n 24 --out-dir {RUNS}

sapienzanlp/Minerva-7B-instruct-v1.0 | GPU Tesla T4 15.6 GB | precisione fp16 | transformers 5.15.0
config.json: 100% 761/761 [00:00<00:00, 2.06MB/s]
tokenizer_config.json: 100% 2.03k/2.03k [00:00<00:00, 784kB/s]
tokenizer.json: 100% 3.67M/3.67M [00:00<00:00, 49.0MB/s]
special_tokens_map.json: 100% 670/670 [00:00<00:00, 2.21MB/s]
dati: train 742 | dev 165
fine turno: id=51202 repr='<|eot_id|>' | eos_token_id del tokenizer=51202 | generate() si fermera' su [51202]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 49.1MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.98G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/9.90G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  17% 2.55G/14.8G [00:57<05:35, 36.5MB/s, 36.5MB/s  ]
Reconstructing (incomplete total...):  26% 3.77G/14.8G [02:07<08:46, 20.9MB/s, 20.9MB/s  ]
Recons

In [11]:
# --- 9b. Adapter prodotto ----------------------------------------------------
import glob, json, os
ADAPTER = sorted(glob.glob(f"{RUNS}/*__T3/adapter_final"))
assert ADAPTER, "adapter non trovato: la cella sopra e' arrivata in fondo?"
ADAPTER = ADAPTER[-1]
print("ADAPTER =", ADAPTER)

s = json.load(open(os.path.join(os.path.dirname(ADAPTER), "summary.json")))
print(f"{s['minuti']} min | best {s['best']['metrica']} = {s['best']['valore']}")
print("andamento:")
for h in s["storia"]:
    print(f"  step {h.get('step'):>4}  loss {h.get('eval_loss', 0):.3f}  "
          f"ctx_delta {h.get('eval_ctx_delta', 0):+.4f}  "
          f"gen_chrf {h.get('eval_gen_chrf', 0):.2f}  "
          f"len {h.get('eval_rapporto_lunghezza', 0):.2f}x")

ADAPTER = /content/runs/Minerva-7B-instruct-v1.0__T3/adapter_final
108.1 min | best ctx_delta = 0.5252
andamento:
  step   23  loss 3.288  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step   46  loss 3.108  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step   69  loss 3.100  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step   92  loss 3.099  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step  115  loss 3.373  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step  138  loss 3.359  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step  161  loss 3.735  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step  184  loss 3.697  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x
  step  188  loss 3.696  ctx_delta +0.0000  gen_chrf 0.00  len 0.00x


## 10 — `t3_eval.py`

La metrica primaria è l'**ablazione**: stesso item generato due volte, col contesto
vero e con uno preso da un altro punto, e chrF++ *fra le due uscite*.

- chrF alto (> 60) o alta frazione di uscite identiche → cieco al contesto
- chrF basso **e** uscite sensate → il contesto entra nella risposta

chrF contro il riferimento resta come colonna descrittiva. Su un task con
riferimento singolo e centinaia di repliche valide non è un criterio di merito.

Include due leve che agiscono **senza riaddestrare**:

- `cad` — decodifica contestuale: $\text{logit} = (1+\gamma)\,\text{logit}(y \mid c) - \gamma\,\text{logit}(y \mid \varnothing)$
- `bon` — best-of-n con selezione reference-free sul guadagno di verosimiglianza dato dal contesto

In [12]:
%%writefile t3_eval.py
#!/usr/bin/env python3
"""
t3_eval.py — valutazione di T3 centrata sulla domanda giusta.

La domanda non e' "quanto assomiglia al turno realmente pronunciato" (riferimento
singolo su un task open-ended: chrF misura la fortuna) ma "l'uscita cambia se
cambia il contesto". Da qui la metrica primaria:

  ABLAZIONE — si genera lo stesso item due volte, col contesto vero e con un
  contesto preso da un altro punto, e si misura il chrF++ FRA LE DUE USCITE.
      chrF alto / identiche       -> il modello e' cieco al contesto
      chrF basso e uscite sensate -> il contesto sta entrando nella risposta

Modalita' di decodifica (--modo):
  greedy      riferimento
  nucleus     top_p 0.9, T 0.8
  cad         decodifica contestuale: logit = (1+g)*logit(y|ctx) - g*logit(y|no ctx).
              Amplifica in inferenza la differenza fra le due distribuzioni.
              Non riaddestra niente e agisce esattamente sul difetto osservato.
  bon         best-of-n: n candidati con nucleus, si tiene quello che massimizza
              logP(cand|ctx) - logP(cand|senza ctx) normalizzato per lunghezza.
              Criterio reference-free: nessuno guarda il target.

Uso:
    python t3_eval.py --model minerva --adapter /kaggle/working/runs/..._T3/adapter_final \
        --split-dir /kaggle/working/split_t3 --modo greedy cad bon --gamma 0.5
"""
from __future__ import annotations

import argparse
import json
import os
import random
import re
import statistics

from t3_train import (base_ambiente, carica_split, carica_token, id_fine_turno,
                      render, resolve_model, slug, sostituisci_contesto)

# Marcatori dialettali: proxy grezzo e dichiarato tale, non un classificatore.
MARCATORI = re.compile(
    r"\b(nun|ca|'a|'o|'e|'nu|na|nnu|chill|chest|accuss|pecch|aggi|sta[cm]|mo'|"
    r"tenimm|simm|jamm|facimm|vulit|vede|guagli|assaje|overo|bbuon|nzomma)",
    re.IGNORECASE)


def dialettalita(testo: str) -> float:
    parole = testo.split()
    if not parole:
        return 0.0
    return sum(1 for p in parole if MARCATORI.match(p)) / len(parole)


def senza_contesto(prompt: str) -> str:
    """Stesso prompt con il blocco di contesto svuotato: e' il ramo 'no ctx'
    della decodifica contestuale."""
    testa, _, istr = prompt.partition("\n---\n")
    return testa.split("\n")[0] + "\n(nessun contesto disponibile)\n---\n" + istr


# --------------------------------------------------------------------------- #
# Generazione
# --------------------------------------------------------------------------- #

def genera_hf(model, tok, prompt, eot_ids, max_new, campiona, temp, top_p, seed=None):
    import torch
    if seed is not None:
        torch.manual_seed(seed)
    inp = tok(render(tok, prompt), add_special_tokens=False,
              return_tensors="pt").to(model.device)
    with torch.no_grad():
        o = model.generate(**inp, max_new_tokens=max_new, do_sample=campiona,
                           temperature=temp if campiona else None,
                           top_p=top_p if campiona else None,
                           eos_token_id=eot_ids, pad_token_id=tok.pad_token_id,
                           repetition_penalty=1.15, no_repeat_ngram_size=3)
    return tok.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def genera_cad(model, tok, prompt, eot_ids, max_new, gamma, temp=0.8, top_p=0.9,
               campiona=False):
    """logit = (1+gamma)*logit(y|ctx) - gamma*logit(y|senza ctx), passo per passo."""
    import torch
    import torch.nn.functional as F

    ids_c = tok(render(tok, prompt), add_special_tokens=False,
                return_tensors="pt")["input_ids"].to(model.device)
    ids_n = tok(render(tok, senza_contesto(prompt)), add_special_tokens=False,
                return_tensors="pt")["input_ids"].to(model.device)
    kv_c = kv_n = None
    prod = []
    with torch.no_grad():
        for passo in range(max_new):
            o_c = model(input_ids=ids_c, past_key_values=kv_c, use_cache=True)
            o_n = model(input_ids=ids_n, past_key_values=kv_n, use_cache=True)
            kv_c, kv_n = o_c.past_key_values, o_n.past_key_values
            lg = (1 + gamma) * o_c.logits[:, -1, :] - gamma * o_n.logits[:, -1, :]
            if campiona:
                p = F.softmax(lg / temp, dim=-1)
                ordinati, idx = p.sort(descending=True)
                taglia = ordinati.cumsum(-1) - ordinati > top_p
                ordinati[taglia] = 0.0
                scelto = idx.gather(-1, ordinati.multinomial(1))
            else:
                scelto = lg.argmax(-1, keepdim=True)
            if scelto.item() in eot_ids:
                break
            prod.append(scelto.item())
            ids_c = ids_n = scelto
    return tok.decode(prod, skip_special_tokens=True).strip()


def punteggio_contesto(model, tok, prompt, candidato, max_len=512):
    """logP(cand|ctx) - logP(cand|senza ctx), normalizzato per token."""
    import torch
    from t3_train import logp_per_sequenza
    valori = []
    with torch.no_grad():
        for prm in (prompt, senza_contesto(prompt)):
            ids = tok(render(tok, prm, candidato), add_special_tokens=False,
                      return_tensors="pt")["input_ids"][:, :max_len].to(model.device)
            n_p = len(tok(render(tok, prm), add_special_tokens=False)["input_ids"])
            lab = ids.clone()
            lab[:, :n_p] = -100
            valori.append(logp_per_sequenza(model(input_ids=ids).logits, lab)[2].item())
    return valori[0] - valori[1]


# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="minerva")
    ap.add_argument("--adapter", default="")
    ap.add_argument("--split-dir", required=True)
    ap.add_argument("--split", default="test")
    ap.add_argument("--out", default="")
    ap.add_argument("--tag", default="")
    ap.add_argument("--modo", nargs="+", default=["greedy", "cad"],
                    choices=["greedy", "nucleus", "cad", "bon"])
    ap.add_argument("--gamma", type=float, default=0.5)
    ap.add_argument("--n-bon", type=int, default=8)
    ap.add_argument("--max-new", type=int, default=32)
    ap.add_argument("--limite", type=int, default=0)
    ap.add_argument("--ablazione-n", type=int, default=64)
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    a.out = a.out or os.path.join(base_ambiente(), "eval")

    import sacrebleu
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    repo = resolve_model(a.model)
    token = carica_token()
    tok = AutoTokenizer.from_pretrained(repo, token=token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    cc = torch.cuda.get_device_capability(0)
    dtype = torch.bfloat16 if cc[0] >= 8 else torch.float16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=dtype)
    try:
        model = AutoModelForCausalLM.from_pretrained(
            repo, dtype=dtype, quantization_config=quant, device_map={"": 0},
            token=token, attn_implementation="eager")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            repo, torch_dtype=dtype, quantization_config=quant, device_map={"": 0},
            token=token, attn_implementation="eager")
    if a.adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, a.adapter)
        print("adapter:", a.adapter)
    else:
        print("NESSUN adapter: e' il modello di base (arm di confronto)")
    model.eval()
    model.config.use_cache = True

    righe = carica_split(a.split_dir, a.split, a.limite or None)
    eot = id_fine_turno(tok, righe[0]["prompt"], righe[0]["target"])
    eot_ids = sorted({eot, tok.eos_token_id} - {None})
    print(f"{len(righe)} item | fine turno {eot_ids}")

    rng = random.Random(a.seed)
    os.makedirs(a.out, exist_ok=True)
    base_tag = a.tag or ("ft" if a.adapter else "base")
    riepilogo = {}

    for modo in a.modo:
        print(f"\n===== {modo} =====")
        preds = []
        for i, r in enumerate(righe):
            if modo == "greedy":
                g = genera_hf(model, tok, r["prompt"], eot_ids, a.max_new, False, None, None)
            elif modo == "nucleus":
                g = genera_hf(model, tok, r["prompt"], eot_ids, a.max_new, True, 0.8, 0.9,
                              seed=a.seed + i)
            elif modo == "cad":
                g = genera_cad(model, tok, r["prompt"], eot_ids, a.max_new, a.gamma)
            else:                                    # best-of-n
                cand = {genera_hf(model, tok, r["prompt"], eot_ids, a.max_new, True,
                                  0.9, 0.95, seed=a.seed + i * 100 + k)
                        for k in range(a.n_bon)}
                cand = [c for c in cand if c] or [""]
                g = max(cand, key=lambda c: punteggio_contesto(model, tok, r["prompt"], c))
            preds.append({"id": r["id"], "prompt": r["prompt"], "riferimento": r["target"],
                          "generato": g, "fonte": r.get("fonte", "")})
            if i % 25 == 0:
                print(f"  {i}/{len(righe)}  {g[:70]!r}")

        ipo = [p["generato"] for p in preds]
        rif = [p["riferimento"] for p in preds]
        m = {
            "n": len(preds),
            "chrf": round(sacrebleu.corpus_chrf(ipo, [rif], word_order=2).score, 3),
            "lunghezza_media": round(statistics.mean(len(h.split()) for h in ipo), 2),
            "lunghezza_umana": round(statistics.mean(len(t.split()) for t in rif), 2),
            "vuote": sum(1 for h in ipo if not h.strip()),
            "dialettalita": round(statistics.mean(dialettalita(h) for h in ipo), 3),
            "dialettalita_umana": round(statistics.mean(dialettalita(t) for t in rif), 3),
            "uscite_distinte": len(set(ipo)),
        }
        m["rapporto_lunghezza"] = round(m["lunghezza_media"] / m["lunghezza_umana"], 3)

        # --- ABLAZIONE: la metrica primaria ---------------------------------
        n_abl = min(a.ablazione_n, len(righe))
        campione = righe[:n_abl]
        veri = [p["generato"] for p in preds[:n_abl]]
        falsi = []
        for i, r in enumerate(campione):
            altro = campione[(i + 1 + rng.randrange(n_abl - 1)) % n_abl]["prompt"]
            p_falso = sostituisci_contesto(r["prompt"], altro)
            if modo == "cad":
                falsi.append(genera_cad(model, tok, p_falso, eot_ids, a.max_new, a.gamma))
            else:
                falsi.append(genera_hf(model, tok, p_falso, eot_ids, a.max_new,
                                       modo != "greedy", 0.8, 0.9, seed=a.seed + i))
        m["ablazione_chrf_vero_vs_falso"] = round(
            sacrebleu.corpus_chrf(falsi, [veri], word_order=2).score, 3)
        m["ablazione_identiche"] = round(
            sum(1 for v, f in zip(veri, falsi) if v.strip() == f.strip()) / n_abl, 3)
        m["ctx_delta"] = round(statistics.mean(
            punteggio_contesto(model, tok, r["prompt"], v)
            for r, v in zip(campione, veri)), 4)

        print(json.dumps(m, indent=2, ensure_ascii=False))
        print("  lettura: ablazione_chrf ALTA (>60) o identiche alto = cieco al contesto;")
        print("           ctx_delta <= 0 = il contesto non aumenta la verosimiglianza.")

        nome = f"{slug(repo)}__T3__{base_tag}__{modo}"
        with open(os.path.join(a.out, nome + ".preds.jsonl"), "w", encoding="utf-8") as f:
            for p in preds:
                f.write(json.dumps(p, ensure_ascii=False) + "\n")
        json.dump(m, open(os.path.join(a.out, nome + ".metrics.json"), "w"),
                  indent=2, ensure_ascii=False)
        riepilogo[modo] = m

    print("\n===== riepilogo =====")
    colonne = ["chrf", "rapporto_lunghezza", "ablazione_chrf_vero_vs_falso",
               "ablazione_identiche", "ctx_delta", "dialettalita"]
    print(f"{'modo':10s}" + "".join(f"{c[:14]:>16s}" for c in colonne))
    for modo, m in riepilogo.items():
        print(f"{modo:10s}" + "".join(f"{m[c]:>16}" for c in colonne))


if __name__ == "__main__":
    main()

Writing t3_eval.py


## 11 — Baseline: il modello di base, senza adapter

**Da fare per prima.** Se il modello di base ha `ablazione_chrf` più bassa del
fine-tuned, il fine-tuning sta *distruggendo* sensibilità al contesto, e nessun
intervento a valle recupera quello che è stato tolto. È l'ipotesi più importante
da falsificare e costa 10 minuti.

In [13]:
# --- 11. Baseline: modello di base ------------------------------------------
!python t3_eval.py --model {MODEL} --split-dir {SPLIT} --split test \
    --modo greedy --limite 96 --ablazione-n 48 --out {EVAL} --tag base

Loading weights: 100% 291/291 [01:04<00:00,  4.51it/s]
NESSUN adapter: e' il modello di base (arm di confronto)
96 item | fine turno [51202]

===== greedy =====
  0/96  "A: tutt'a bbona cosa, grazzie, sii sii"
  25/96  'Ah, perfetto! Grazie assai.'
  50/96  'A: mmmhhh, si certo, come vuoi tu!'
  75/96  'C: E niente.'
{
  "n": 96,
  "chrf": 10.373,
  "lunghezza_media": 16.8,
  "lunghezza_umana": 8.09,
  "vuote": 0,
  "dialettalita": 0.044,
  "dialettalita_umana": 0.087,
  "uscite_distinte": 94,
  "rapporto_lunghezza": 2.077,
  "ablazione_chrf_vero_vs_falso": 15.829,
  "ablazione_identiche": 0.0,
  "ctx_delta": 1.7601
}
  lettura: ablazione_chrf ALTA (>60) o identiche alto = cieco al contesto;
           ctx_delta <= 0 = il contesto non aumenta la verosimiglianza.

===== riepilogo =====
modo                  chrf  rapporto_lungh  ablazione_chrf  ablazione_iden       ctx_delta    dialettalita
greedy              10.373           2.077          15.829             0.0          1.7601       

## 12 — Valutazione dell'adapter, quattro decodifiche

`greedy` e `nucleus` sono confronti appaiati; `cad` e `bon` sono le due leve di
inferenza. Circa 25–40 minuti in tutto sul test intero.

In [14]:

# --- 12. Valutazione dell'adapter -------------------------------------------
!python t3_eval.py --model {MODEL} --adapter {ADAPTER} --split-dir {SPLIT} \
    --split test --modo greedy nucleus cad bon \
    --gamma 0.5 --n-bon 8 --out {EVAL} --tag ft

Loading weights: 100% 291/291 [01:04<00:00,  4.51it/s]
adapter: /content/runs/Minerva-7B-instruct-v1.0__T3/adapter_final
168 item | fine turno [51202]

===== greedy =====
  0/168  "ma quinni nun è ca s'è magnato 'o puparuolo?"
  25/168  "ma 'nfatti pure chisto è piccerillo"
  50/168  "pucci, cioè è nu modo pe' spennere 'o sordo"
  75/168  "e quinni se faje 'a dummeneca ca te vene 'ncopp'â Germania a piglià 'o"
  100/168  'ah, è nu posto bellissimo'
  125/168  "ma sì, chello c' 'o puó jì a penzà"
  150/168  "no no, ma è assaje tardi, pe' ghì sopra"
{
  "n": 168,
  "chrf": 10.563,
  "lunghezza_media": 8.75,
  "lunghezza_umana": 7.27,
  "vuote": 0,
  "dialettalita": 0.102,
  "dialettalita_umana": 0.084,
  "uscite_distinte": 165,
  "rapporto_lunghezza": 1.204,
  "ablazione_chrf_vero_vs_falso": 12.702,
  "ablazione_identiche": 0.0,
  "ctx_delta": 1.0478
}
  lettura: ablazione_chrf ALTA (>60) o identiche alto = cieco al contesto;
           ctx_delta <= 0 = il contesto non aumenta la verosim

## 13 — Taratura di gamma, sul **dev**

`gamma` troppo alto rende il testo strano: amplifica anche il rumore della
differenza fra le due distribuzioni. Il ginocchio si cerca sul dev.

In [15]:
# --- 13. Sweep di gamma sul dev ---------------------------------------------
for g in (0.0, 0.3, 0.5, 0.8, 1.2):
    print(f"\n########## gamma = {g} ##########")
    cmd = (f"python t3_eval.py --model {MODEL} --adapter {ADAPTER} "
           f"--split-dir {SPLIT} --split dev --modo cad --gamma {g} "
           f"--limite 64 --ablazione-n 48 --out {EVAL} --tag g{g}")
    !{cmd}


########## gamma = 0.0 ##########
Loading weights: 100% 291/291 [01:03<00:00,  4.59it/s]
adapter: /content/runs/Minerva-7B-instruct-v1.0__T3/adapter_final
64 item | fine turno [51202]

===== cad =====
  0/64  "no, è 'o stesso, è 'o stesso però senz' 'o pi~, senz' 'o sarma, senz' "
  25/64  'mh, me piace assaje, me piace assaje'
  50/64  "no, 'o purtammo"
{
  "n": 64,
  "chrf": 9.772,
  "lunghezza_media": 6.94,
  "lunghezza_umana": 7.72,
  "vuote": 0,
  "dialettalita": 0.112,
  "dialettalita_umana": 0.097,
  "uscite_distinte": 64,
  "rapporto_lunghezza": 0.899,
  "ablazione_chrf_vero_vs_falso": 11.385,
  "ablazione_identiche": 0.0,
  "ctx_delta": 1.2165
}
  lettura: ablazione_chrf ALTA (>60) o identiche alto = cieco al contesto;
           ctx_delta <= 0 = il contesto non aumenta la verosimiglianza.

===== riepilogo =====
modo                  chrf  rapporto_lungh  ablazione_chrf  ablazione_iden       ctx_delta    dialettalita
cad                  9.772           0.899          11.385 

## 14 — Tabella comparativa di tutti gli arm

In [16]:
# --- 14. Tabella finale ------------------------------------------------------
import glob, json, os
import pandas as pd

righe = []
for p in sorted(glob.glob(f"{EVAL}/*.metrics.json")):
    m = json.load(open(p))
    nome = os.path.basename(p).replace(".metrics.json", "")
    parti = nome.split("__")
    righe.append(dict(arm=parti[-2] if len(parti) > 2 else nome, modo=parti[-1],
                      chrf=m["chrf"], len_ratio=m["rapporto_lunghezza"],
                      abl_chrf=m["ablazione_chrf_vero_vs_falso"],
                      abl_uguali=m["ablazione_identiche"],
                      ctx_delta=m["ctx_delta"], dial=m["dialettalita"],
                      dial_umana=m["dialettalita_umana"], distinte=m["uscite_distinte"]))
df = pd.DataFrame(righe).sort_values("abl_chrf")
display(df)

print("""
Come leggerla, in quest'ordine:
  1. abl_chrf   PIU' BASSA = usa di piu' il contesto. E' il criterio.
  2. ctx_delta  > 0 e il piu' alto possibile.
  3. len_ratio  vicino a 1.0: se e' 2-3x il fine turno non e' stato imparato
                e le altre colonne non sono interpretabili.
  4. dial       confrontata con dial_umana: molto sopra = iper-dialettalita',
                cioe' il modello sostituisce forme marcate dove non trova la
                parola giusta. E' semanticamente distruttivo, non estetico.
  5. chrf       descrittiva. Non selezionare su questa.
""")

,arm,modo,chrf,len_ratio,abl_chrf,abl_uguali,ctx_delta,dial,dial_umana,distinte
9,g1.2,cad,8.758,0.784,7.186,0.0,2.7778,0.083,0.097,64
7,g0.5,cad,10.270,0.917,7.955,0.0,1.9313,0.125,0.097,64
8,g0.8,cad,8.767,0.810,8.191,0.0,2.5184,0.093,0.097,63
2,ft,cad,10.413,0.857,8.427,0.0,2.3849,0.070,0.084,160
6,g0.3,cad,10.517,0.878,9.412,0.0,1.6599,0.120,0.097,64
5,g0.0,cad,9.772,0.899,11.385,0.0,1.2165,0.112,0.097,64
1,ft,bon,9.408,0.938,11.880,0.0,1.6082,0.090,0.084,168
3,ft,greedy,10.563,1.204,12.702,0.0,1.0478,0.102,0.084,165
4,ft,nucleus,10.076,1.151,12.707,0.0,0.9921,0.093,0.084,167
0,base,greedy,10.373,2.077,15.829,0.0,1.7601,0.044,0.087,94



Come leggerla, in quest'ordine:
  1. abl_chrf   PIU' BASSA = usa di piu' il contesto. E' il criterio.
  2. ctx_delta  > 0 e il piu' alto possibile.
  3. len_ratio  vicino a 1.0: se e' 2-3x il fine turno non e' stato imparato
                e le altre colonne non sono interpretabili.
  4. dial       confrontata con dial_umana: molto sopra = iper-dialettalita',
                cioe' il modello sostituisce forme marcate dove non trova la
                parola giusta. E' semanticamente distruttivo, non estetico.
  5. chrf       descrittiva. Non selezionare su questa.



In [17]:
# --- 14b. Ispezione qualitativa: 12 item, contesto vero e uscita ------------
import json, glob, random
f = sorted(glob.glob(f"{EVAL}/*ft__cad.preds.jsonl"))
if not f:
    f = sorted(glob.glob(f"{EVAL}/*ft__greedy.preds.jsonl"))
righe = [json.loads(l) for l in open(f[-1], encoding="utf-8")]
for r in random.Random(0).sample(righe, min(12, len(righe))):
    ctx = r["prompt"].split("\n---\n")[0].split("\n")[1:]
    print("-" * 74)
    print("\n".join(ctx[-3:]))
    print(f"  GENERATO: {r['generato']}")
    print(f"  UMANO   : {r['riferimento']}")

--------------------------------------------------------------------------
B: simmo juti c' 'a chiavetta 'n copisteria
A: sì
A: e e iss~ e chillo llà ha visto tutte 'e toje
  GENERATO: ah, te fa "mah, ma tanto nun ll'avimmo c~ c~ cchiammà pe' sfenecere"
  UMANO   : no no no, era, era 'n bolognina
--------------------------------------------------------------------------
C: ha ditto "sì sì, si vulite ve 'e faccio 'n hd"
C: però ll'aggi'a sistemà tutte
C: quinni ce vularrà nu poco
  GENERATO: no vabbuò, tanto nun è hd
  UMANO   : e vuje ce avite ditto "ma scusate, avite visto 'e ffoto"
--------------------------------------------------------------------------
C: ah okay, okay
B: sì sì sì
A: nun campa 'n Italia 'a nu poco
  GENERATO: nun campa 'n Italia 'a nu poco
  UMANO   : cchiù o meno
--------------------------------------------------------------------------
A: che cosa
A: ah
A: ah
  GENERATO: è nu sfizio
  UMANO   : cioè ner senso si tu 'o tuocche e po' te miette 'a mana 'mmocca, oka

## 15 — Salvare gli adapter

Il disco è temporaneo in entrambi gli ambienti e i run qui durano ore: se non
salvi, alla disconnessione perdi tutto.

**Kaggle** — *Save Version > Save & Run All* persiste `/kaggle/working`; da lì
l'output torna montabile come Dataset. Per riprendere un run interrotto, allegalo
e rilancia con `--resume-dir /kaggle/input/<nome-output>/runs`.

**Colab** — la sessione si scollega senza preavviso, quindi conviene scrivere
direttamente su Drive:

```python
from google.colab import drive; drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/nap-t3/runs"   # da rieseguire PRIMA del training
```

In alternativa, o in aggiunta, il backup sull'Hub qui sotto: è l'unico che
sopravvive alla morte della macchina in tutti e due i casi.

In [18]:
# --- 15. Backup dell'adapter sull'Hub (facoltativo) -------------------------
# from huggingface_hub import HfApi
# api = HfApi(token=os.environ["HF_TOKEN"])
# REPO = "TUO_UTENTE/minerva-nap-t3"
# api.create_repo(REPO, private=True, exist_ok=True)
# api.upload_folder(folder_path=ADAPTER, repo_id=REPO, path_in_repo="adapter_final")
# print("caricato su", REPO)

## Cosa concludere, e in che ordine

1. **Baseline contro fine-tuned sull'ablazione.** Se il base è più sensibile al
   contesto, il problema è il fine-tuning, non il prompt né la decodifica.
2. **`--lambda-ctx 0.5` contro `0`.** Stesso split, stesso seme, stesso numero di
   step: l'unica variabile è la loss contrastiva. È l'esperimento che risponde
   alla domanda del progetto.
3. **`--finestra 6` contro `--finestra 3`.** Isola il contributo del contesto
   ricostruito da quello dell'obiettivo.
4. **`cad` e `bon`.** Guadagno a costo zero di training: vanno riportati come
   interventi di inferenza, distinti dal merito del modello.

Ogni arm cambia **una** cosa. Con n = 2 conversazioni, differenze piccole non sono
interpretabili: riporta gli intervalli, non i punti.